<a href="https://colab.research.google.com/github/Eleonwra/elcardiocc-baseline-ner/blob/main/preprocessing/split_dataset_sentences.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets

In [ ]:
import pickle
with open('final_dataset.pickle', 'rb') as data:
    final_dataset = pickle.load(data)

##Split by section

In [ ]:
import re
def split_text_by_titles(text, titles):
    title_pattern = re.compile('|'.join([title for title in titles]))
    start_title_positions = [match.start() for match in re.finditer(title_pattern, text)]
    title_positions = start_title_positions + [len(text)]
    sections = [text[title_positions[i]:title_positions[i+1]] for i in range(len(title_positions)-1)]
    return sections, title_positions
titles = ['ΕΝΗΜΕΡΩΤΙΚΟ ΣΗΜΕΙΩΜΑ',
'ΣΤΟΙΧΕΙΑ ΑΣΘΕΝΟΥΣ',
'ΙΣΤΟΡΙΚΟ – ΑΝΤΙΚΕΙΜΕΝΙΚΗ ΕΞΕΤΕΤΑΣΗ',
'ΠΟΡΕΙΑ ΝΟΣΟΥ',
'ΔΙΑΓΝΩΣΗ ΕΞΟΔΟΥ',
'ΘΕΡΑΠΕΥΤΙΚΗ ΑΓΩΓΗ - ΕΠΕΜΒΑΣΕΙΣ',
'ΟΔΗΓΙΕΣ ΚΑΤΑ ΤΗΝ ΕΞΟΔΟ - ΠΑΡΑΤΗΡΗΣΕΙΣ',
'ΕΡΓΑΣΤΗΡΙΑΚΕΣ ΕΞΕΤΑΣΕΙΣ',
'ΕΞΕΤΑΣΕΙΣ',
'Λοιπές εξετάσεις',
'ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΑΝΤΙΚΕΙΜΕΝΙΚΗ ΕΞΕΤΕΤΑΣΗ - ΙΣΤΟΡΙΚΟ',
'ΠΑΡΑΚΛΙΝΙΚΕΣ ΕΞΕΤΑΣΕΙΣ',
'ΙΣΤΟΡΙΚΟ – ΑΝΤΙΚΕΙΜΕΝΙΚΗ ΕΞΕΤΑΣΗ'
]

In [ ]:
text = ''.join(final_dataset['train'][0]['tokens'])
splited, title_positions = split_text_by_titles(text, titles)

In [ ]:
from datasets import Features, Sequence, Value, ClassLabel, Dataset
for split in final_dataset.keys():
    sections_tokens_list = []
    sections_ner_tags_list = []
    patient_id_list = []
    for i, example in enumerate(final_dataset[split]):
        character_tokens = example['tokens']
        text = "".join(character_tokens)
        sections, title_positions =  split_text_by_titles(text,titles)
        sections_tokens_list.append([list(section) for section in sections])
        sections_ner_tags = []
        ner_tags = example['ner_tags']
        for i, section in enumerate(sections):
          start = title_positions[i]
          end = title_positions[i + 1]
          sections_ner_tags.append(ner_tags[start:end])
          if len(section) != len(ner_tags[start:end]):
              print("tokens length", len(section))
              print("ner_tags legnth:", len(ner_tags[start:end]))
              print('difference in alignment')
        sections_ner_tags_list.append(sections_ner_tags)
        patient_id_list.append(example["patient_id"])
        if len(sections_tokens_list[-1]) != len(sections_ner_tags_list[-1]):
          print(f"Example {i + 1}:")
          print("sections_tokens_list length:", len(sections_tokens_list[-1]))
          print("sections_ner_tags_list length:", len(sections_ner_tags_list[-1]))

    if len(sections_tokens_list) != len(sections_ner_tags_list):
        print("Final lengths:")
        print("sections_tokens_list length:", len(sections_tokens_list))
        print("sections_ner_tags_list length:", len(sections_ner_tags_list))
        print("patient_id_list length:", len(patient_id_list))
        print()
    features=Features({
        "patient_id": Value(dtype='int32', id=None),
        "sections_tokens": Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None),
        "sections_ner_tags": Sequence(feature=Sequence(feature=ClassLabel(names=["O", "B-ENTITY",'I-ENTITY'], id=None), length=-1, id=None), length=-1, id=None),
    })
    final_dataset[split] = Dataset.from_dict(
    {'sections_tokens': sections_tokens_list,'sections_ner_tags':sections_ner_tags_list, "patient_id":  patient_id_list},features=features)

In [ ]:
with open('final_dataset_sections.pickle', 'wb') as output:
    pickle.dump(final_dataset, output)

##Split by sentence

In [ ]:
!pip install stanza

In [ ]:
import stanza
stanza.download('el')
nlp_stanza = stanza.Pipeline('el')

In [ ]:
nlp_stanza = stanza.Pipeline('el')
def tokenize_text_with_indices(text):
    doc = nlp_stanza(text)
    tokenized_sentences = []
    for sentence in doc.sentences:
        pattern = re.escape(sentence.text)
        match = re.search(pattern, text)
        if match:
            start_index = match.start()
            end_index = match.end()
            sentence_info = {
                'text': sentence.text,
                'start_index': start_index,
                'end_index': end_index
            }
            tokenized_sentences.append(sentence_info)
    return tokenized_sentences

In [ ]:
from datasets import Features, Sequence, Value, ClassLabel, Dataset
for split in final_dataset.keys():
    sentences_tokens_list = []
    sentences_ner_tags_list = []
    patient_id_list = []
    extra_info_list = []
    for j, example in enumerate(final_dataset[split]):
        doc_sentences_ner_tags_list = []
        doc_patient_id_list = []
        doc_extra_info_list = []
        doc_sentences_tokens_list = []
        for i in range(len(example['sections_tokens'])):
          character_tokens = example['sections_tokens'][i]
          text = "".join(character_tokens)
          sentences_info =  tokenize_text_with_indices(text)
          sentences = [info['text'] for info in sentences_info]
          doc_sentences_tokens_list.append([list(sentence) for sentence in sentences])
          sentences_ner_tags = []
          ner_tags = example['sections_ner_tags'][i]

          for info in sentences_info:
              start_index = info['start_index']
              end_index = info['end_index']
              sentence_ner_tags = ner_tags[start_index:end_index]
              sentences_ner_tags.append(sentence_ner_tags)
          doc_sentences_ner_tags_list.append(sentences_ner_tags)
          if len(doc_sentences_ner_tags_list) != len(doc_sentences_tokens_list):
            print(f"Example {j+ 1}:")
            print("sentences_tokens_list length:", len(doc_sentences_ner_tags_list))
            print("sentence_ner_tags_list length:", len(doc_sentences_tokens_list))
            print('patient id ',doc_patient_id_list[-1])

          if len(doc_sentences_ner_tags_list[-1]) != len(doc_sentences_tokens_list[-1]):
            print(f"Example {j+ 1}:")
            print("sentences_tokens_list length:", len(doc_sentences_ner_tags_list[-1]))
            print("sentence_ner_tags_list length:", len(doc_sentences_tokens_list[-1]))
            print('patient id ',doc_patient_id_list[-1])
        doc_sentences_tokens_list = [item for sublist in doc_sentences_tokens_list for item in sublist]
        doc_sentences_ner_tags_list = [item for sublist in doc_sentences_ner_tags_list for item in sublist]
        if len(doc_sentences_ner_tags_list[-1]) != len(doc_sentences_tokens_list[-1]):
            print(f"Example {j+ 1}:")
            print("sentences_tokens_list length:", len(doc_sentences_ner_tags_list[-1]))
            print("sentence_ner_tags_list length:", len(doc_sentences_tokens_list[-1]))
            print('patient id ',doc_patient_id_list[-1])
        if len(doc_sentences_tokens_list) != len(doc_sentences_ner_tags_list):
            print(len(doc_sentences_tokens_list))
            print(len(doc_sentences_ner_tags_list))
            print('error')
        if len(doc_patient_id_list) != len(doc_extra_info_list):
            print('error')
        sentences_tokens_list.append(doc_sentences_tokens_list)
        sentences_ner_tags_list.append(doc_sentences_ner_tags_list)
        patient_id_list.append(example["patient_id"])
    features=Features({
        "patient_id": Value(dtype='int32', id=None),
        "sentences_tokens": Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None),
        "sentences_ner_tags": Sequence(feature=Sequence(feature=ClassLabel(names=["O", "B-ENTITY",'I-ENTITY'], id=None), length=-1, id=None), length=-1, id=None),
    })
    final_dataset[split] = Dataset.from_dict(
    {'sentences_tokens': sentences_tokens_list,'sentences_ner_tags': sentences_ner_tags_list, "patient_id":  patient_id_list},features=features)

In [ ]:
import os
with open('final_dataset_sentences.pickle', 'wb') as output:
    pickle.dump(final_dataset, output)